# EGN 321 — Module 7 Assignment 7.2 Complete Instructor Demo
## Failure Investigation, Recovery, Ollama Review, and Report Export

This notebook is designed for the **instructor to demo the entire Assignment 7.2 workflow** in class.

It creates a controlled failure, analyzes structured logs, builds the required evidence table, identifies the first observable symptom, proposes two causes, runs deterministic verification tests, performs recovery, uses Ollama as an evidence reviewer, generates the required failure report, and exports all supporting documents in one ZIP.

### Required report sections produced
1. System Before Failure  
2. Failure Introduced  
3. First Observable Symptom  
4. Log Evidence  
5. Timeline of Events  
6. Possible Cause 1  
7. Possible Cause 2  
8. Recovery Procedure  
9. Verification of Recovery  
10. What the Logs Told Me  
11. What the Logs Did Not Tell Me  
12. Improvement Made  
13. Remaining Limitations


## Demo Flow

```text
Healthy System
    ↓
Controlled Failure
    ↓
Structured Logs
    ↓
Required Evidence Table
    ↓
First Observable Symptom
    ↓
Possible Causes
    ↓
Deterministic Verification Tests
    ↓
Recovery
    ↓
Recovery Verification
    ↓
Ollama Review / Red Team
    ↓
Failure Report
    ↓
README + AI_LOG + ZIP
```

**Important:** logs and tests establish evidence. Ollama reviews, challenges, and helps draft from that evidence.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import pandas as pd
import sqlite3
import json
import shutil
import zipfile
import os

BASE = Path("/content/module7_assignment7_2_instructor_demo")
BASE.mkdir(exist_ok=True)

LOG_FILE = BASE / "module7_app.jsonl"
DB_FILE = BASE / "module7_demo.db"
OFFLINE_DB = BASE / "module7_demo.db.offline"
EXPORT_DIR = BASE / "assignment7_2_export"
EXPORT_DIR.mkdir(exist_ok=True)

for p in [LOG_FILE, DB_FILE, OFFLINE_DB]:
    if p.exists():
        p.unlink()

def log_event(level, event, component, **fields):
    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "level": level,
        "component": component,
        "event": event,
        **fields
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")
    return record

print("Demo workspace:", BASE)


## 1. System Before Failure

Create a healthy SQLite database, store readings, and record normal operational logs.


In [ ]:
conn = sqlite3.connect(DB_FILE)

conn.execute("""
CREATE TABLE readings (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp TEXT,
    temperature_c REAL,
    status TEXT
)
""")

base_time = datetime.now(timezone.utc)

for i in range(1, 21):
    ts = (base_time + timedelta(seconds=i*2)).isoformat()
    value = 23.0 + (i * 0.03)

    conn.execute(
        "INSERT INTO readings(timestamp, temperature_c, status) VALUES(?,?,?)",
        (ts, value, "OK")
    )

    log_event(
        "INFO",
        "reading_stored",
        "sensor",
        sample_number=i,
        temperature_c=round(value, 3),
        sensor_status="OK",
        health_state="HEALTHY"
    )

conn.commit()
conn.close()

log_event(
    "INFO",
    "system_baseline",
    "system",
    database_available=True,
    dashboard_available=True,
    sensor_available=True,
    health_state="HEALTHY"
)

print("Healthy baseline created.")


## 2. Failure Introduced

For the demo, the database file is renamed. This simulates a realistic **database unavailable / wrong path** failure while keeping the original database recoverable.


In [ ]:
shutil.move(DB_FILE, OFFLINE_DB)

log_event(
    "WARNING",
    "failure_injected",
    "storage",
    failure_type="database_unavailable",
    action="database file renamed",
    expected_effect="application cannot access expected readings table",
    health_state="DEGRADED"
)

try:
    conn = sqlite3.connect(DB_FILE)

    # SQLite may create an empty file automatically, so force a query
    # that requires the expected application schema.
    conn.execute("SELECT COUNT(*) FROM readings").fetchone()
    conn.close()

except Exception as exc:
    log_event(
        "ERROR",
        "database_read_failed",
        "storage",
        database_path=str(DB_FILE),
        exception_type=type(exc).__name__,
        exception_message=str(exc),
        health_state="FAILED"
    )

print("Controlled database failure introduced.")


## 3. Load and Parse the Logs

This is where the investigation starts. The report should be built from evidence, not from memory.


In [ ]:
records = []

with open(LOG_FILE, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

logs = pd.DataFrame(records)
logs["timestamp"] = pd.to_datetime(logs["timestamp"], utc=True)
logs = logs.sort_values("timestamp").reset_index(drop=True)

print("Log rows:", len(logs))
print("Time range:", logs["timestamp"].min(), "to", logs["timestamp"].max())

display(
    logs[["timestamp","level","component","event","health_state"]]
    .tail(12)
)


## 4. Required Evidence Table

The assignment requires students to convert logs into a simple evidence table:

| Time | Component | Event | Evidence | Interpretation |
|---|---|---|---|---|

This forces the investigation to separate **what happened** from **what the student thinks it means**.


In [ ]:
def interpretation_for_event(row):
    event = row["event"]

    if event == "system_baseline":
        return "System confirmed healthy before the controlled failure."
    if event == "failure_injected":
        return "Expected database path was intentionally made unavailable."
    if event == "database_read_failed":
        return "Application could not access the expected readings table."
    if event == "recovery_action":
        return "Recovery action attempted to restore expected database path."
    if event == "recovery_verified":
        return "Database, schema, and preserved data were verified after recovery."
    if event == "logging_improvement_added":
        return "Additional evidence fields were added for future investigations."

    return "Operational event; review context before drawing a conclusion."

evidence_table = logs.copy()

evidence_table["Time"] = evidence_table["timestamp"].dt.strftime("%H:%M:%S")
evidence_table["Component"] = evidence_table["component"]
evidence_table["Event"] = evidence_table["event"]
evidence_table["Evidence"] = evidence_table["level"].map(
    lambda x: f"{x} log entry"
)
evidence_table["Interpretation"] = evidence_table.apply(
    interpretation_for_event,
    axis=1
)

required_evidence = evidence_table[
    ["Time","Component","Event","Evidence","Interpretation"]
].copy()

display(required_evidence.tail(10))


## 5. First Observable Symptom

Do not automatically call the first logged symptom the root cause.

The correct wording is:

> **The first observable symptom in the available evidence was...**

That keeps the report evidence-based.


In [ ]:
problem_logs = logs[
    logs["level"].isin(["WARNING","ERROR","ALERT"])
].copy()

first_problem = problem_logs.iloc[0]

print("First observable warning/error:")
display(
    first_problem[
        ["timestamp","level","component","event","health_state"]
    ].to_frame("value")
)

FIRST_OBSERVABLE = first_problem["event"]

print("\nReport wording:")
print(
    f"The first observable symptom in the available logs was "
    f"'{FIRST_OBSERVABLE}'. This does not by itself prove the final root cause."
)


## 6. Build the Failure Timeline


In [ ]:
timeline_events = [
    "system_baseline",
    "failure_injected",
    "database_read_failed"
]

failure_timeline = logs[
    logs["event"].isin(timeline_events)
][
    ["timestamp","level","component","event","health_state"]
].copy()

display(failure_timeline)


## 7. Possible Cause 1 and Verification Test

### Possible Cause 1
The expected database path became unavailable or changed.

### Verification Test
Check whether the expected database path exists.


In [ ]:
def verify_database_path(path):
    p = Path(path)

    return {
        "test": "database_path_exists",
        "database_path": str(p),
        "exists": p.exists()
    }

cause1_before = verify_database_path(DB_FILE)

print(json.dumps(cause1_before, indent=2))


## 8. Possible Cause 2 and Verification Test

### Possible Cause 2
A database file exists, but the expected `readings` table is missing or invalid.

### Verification Test
Inspect SQLite metadata and confirm that the required table exists.


In [ ]:
def verify_database_schema(path):
    p = Path(path)

    result = {
        "test": "database_schema_has_readings_table",
        "database_path": str(p),
        "exists": p.exists(),
        "has_readings_table": False,
        "tables_found": [],
        "error": None
    }

    if not p.exists():
        return result

    try:
        conn = sqlite3.connect(p)

        tables = conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        ).fetchall()

        conn.close()

        names = [row[0] for row in tables]

        result["tables_found"] = names
        result["has_readings_table"] = "readings" in names

    except Exception as exc:
        result["error"] = f"{type(exc).__name__}: {exc}"

    return result

cause2_before = verify_database_schema(DB_FILE)

print(json.dumps(cause2_before, indent=2))


## 9. Recovery Procedure

Restore the original database, then verify:
- expected path;
- schema;
- preserved rows;
- final health state.


In [ ]:
# SQLite may have created a blank database at the expected path.
if DB_FILE.exists():
    DB_FILE.unlink()

shutil.move(OFFLINE_DB, DB_FILE)

log_event(
    "INFO",
    "recovery_action",
    "storage",
    action="restored original database file to expected path",
    database_path=str(DB_FILE),
    health_state="RECOVERING"
)

cause1_after = verify_database_path(DB_FILE)
cause2_after = verify_database_schema(DB_FILE)

conn = sqlite3.connect(DB_FILE)

rows_preserved = conn.execute(
    "SELECT COUNT(*) FROM readings"
).fetchone()[0]

conn.close()

recovery_passed = (
    cause1_after["exists"]
    and cause2_after["has_readings_table"]
    and rows_preserved == 20
)

log_event(
    "INFO",
    "recovery_verified",
    "system",
    database_available=cause1_after["exists"],
    schema_valid=cause2_after["has_readings_table"],
    rows_preserved=rows_preserved,
    recovery_passed=recovery_passed,
    health_state="HEALTHY" if recovery_passed else "FAILED"
)

print("Recovery passed:", recovery_passed)
print("Rows preserved:", rows_preserved)

assert recovery_passed


## 10. What the Logs Told Me / Did Not Tell Me

A strong failure report includes both.

### The logs told us
- the system was healthy before the failure;
- the expected database became unavailable;
- access to the required table failed;
- recovery was attempted;
- recovery verification passed.

### The logs did not prove
- who or what changed the path in a real incident;
- whether permissions changed;
- whether the disk was full;
- whether another OS/service event occurred first;
- the ultimate root cause outside the application's own evidence.


In [ ]:
what_logs_told = [
    "System had a healthy baseline.",
    "Database access failed after the controlled failure.",
    "Expected application schema was unavailable at the expected path.",
    "Recovery restored path, schema, and original row count."
]

what_logs_did_not_tell = [
    "Who or what changed the database path in a real incident.",
    "Whether disk pressure contributed.",
    "Whether permissions changed.",
    "Whether an external OS/service event occurred first."
]

print("WHAT THE LOGS TOLD ME")
for item in what_logs_told:
    print(" -", item)

print("\nWHAT THE LOGS DID NOT TELL ME")
for item in what_logs_did_not_tell:
    print(" -", item)


## 11. Improvement Made

The investigation should improve the system.

This demo adds a log entry documenting the fields that should be captured next time.


In [ ]:
log_event(
    "INFO",
    "logging_improvement_added",
    "logging",
    improvement=(
        "Future storage failures should include component, database_path, "
        "exception_type, health_state, attempted_recovery, and verification_result."
    ),
    health_state="HEALTHY"
)

print("Logging improvement recorded.")


# 12. Ollama — Evidence Review and Red Team

Ollama is used **after** deterministic evidence is established.

### Ollama may
- summarize log evidence;
- identify plausible causes;
- identify missing evidence;
- propose verification tests;
- challenge a conclusion;
- help draft the report.

### Ollama may not
- invent log events;
- invent sensor readings;
- change deterministic test results;
- claim an unverified root cause is proven.


In [ ]:
# Install and start Ollama in Colab
import subprocess
import shutil as _shutil
import time as _time
import requests

def run_shell(cmd):
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        capture_output=True
    )

    if result.stdout:
        print(result.stdout[-2500:])

    if result.returncode != 0:
        print(result.stderr[-2500:])

    return result.returncode

run_shell("apt-get update -qq")
run_shell("apt-get install -y -qq curl zstd")

if _shutil.which("ollama") is None:
    run_shell("curl -fsSL https://ollama.com/install.sh | sh")

OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_MODEL = "qwen2.5:0.5b"

def ollama_running():
    try:
        return requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=3
        ).status_code == 200
    except requests.RequestException:
        return False

if not ollama_running():
    log = open("/tmp/ollama_module7.log", "a")

    proc = subprocess.Popen(
        ["ollama","serve"],
        stdout=log,
        stderr=subprocess.STDOUT
    )

    for _ in range(30):
        if ollama_running():
            break
        _time.sleep(1)

if not ollama_running():
    raise RuntimeError("Ollama did not start.")

run_shell(f"ollama pull {OLLAMA_MODEL}")

print("Ollama ready:", OLLAMA_MODEL)


In [ ]:
AI_SYSTEM_POLICY = '''
You are an engineering-software incident-review assistant for EGN 321.

Use only evidence supplied in the prompt.

Rules:
1. Do not invent events, timestamps, readings, files, causes, or test results.
2. Separate observation, inference, and hypothesis.
3. Never claim a root cause is proven unless deterministic evidence proves it.
4. For every possible cause include:
   - supporting evidence,
   - missing evidence,
   - deterministic verification test.
5. Challenge conclusions that go beyond the evidence.
6. If evidence is insufficient, say exactly what evidence is missing.
'''

def ask_ollama(prompt, timeout=600, num_predict=700):
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "system": AI_SYSTEM_POLICY,
        "stream": False,
        "keep_alive": "30m",
        "options": {
            "temperature": 0.2,
            "num_predict": num_predict,
            "num_ctx": 4096
        }
    }

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json=payload,
        timeout=(10, timeout)
    )

    response.raise_for_status()
    return response.json()["response"].strip()

print("Warm-up:", ask_ollama("Reply with exactly READY.", num_predict=10))


## 13. Build the Deterministic Evidence Package for Ollama


In [ ]:
# Re-read final logs after recovery and improvement.
records = []

with open(LOG_FILE, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

logs = pd.DataFrame(records)
logs["timestamp"] = pd.to_datetime(logs["timestamp"], utc=True)
logs = logs.sort_values("timestamp").reset_index(drop=True)

evidence = {
    "first_observable_symptom": FIRST_OBSERVABLE,
    "cause_1_test_before_recovery": cause1_before,
    "cause_2_test_before_recovery": cause2_before,
    "cause_1_test_after_recovery": cause1_after,
    "cause_2_test_after_recovery": cause2_after,
    "rows_preserved_after_recovery": int(rows_preserved),
    "recovery_passed": bool(recovery_passed),
    "what_logs_told": what_logs_told,
    "what_logs_did_not_tell": what_logs_did_not_tell,
    "event_counts": logs["event"].value_counts().to_dict()
}

print(json.dumps(evidence, indent=2))


## 14. Ollama Investigator Review


In [ ]:
investigator_prompt = f'''
Here is deterministic incident evidence:

{json.dumps(evidence, indent=2)}

Return exactly these sections:

1. What the evidence proves
2. What the evidence does NOT prove
3. Possible Cause 1
   - supporting evidence
   - missing evidence
   - deterministic verification test
4. Possible Cause 2
   - supporting evidence
   - missing evidence
   - deterministic verification test
5. Recommended logging improvement

Do not invent events.
Do not call a hypothesis proven unless the deterministic evidence proves it.
'''

ollama_investigation = ask_ollama(
    investigator_prompt,
    num_predict=900
)

print(ollama_investigation)


## 15. Ollama Red-Team Review

Now ask Ollama to challenge a conclusion that sounds reasonable but may overstate the evidence.


In [ ]:
provisional_conclusion = (
    "The database failure was caused by a corrupted database file."
)

red_team_prompt = f'''
Deterministic evidence:

{json.dumps(evidence, indent=2)}

Provisional conclusion:
{provisional_conclusion}

Red-team this conclusion.

Return:
- what the evidence actually supports;
- what the conclusion overstates;
- alternative explanations;
- the strongest next deterministic test.

Do not invent new evidence.
'''

ollama_red_team = ask_ollama(
    red_team_prompt,
    num_predict=700
)

print(ollama_red_team)


## 16. Generate the Required Evidence Table Again After Recovery


In [ ]:
def interpretation_for_event(row):
    event = row["event"]

    mapping = {
        "system_baseline": "System confirmed healthy before failure.",
        "failure_injected": "Expected database path intentionally made unavailable.",
        "database_read_failed": "Application could not access required database schema.",
        "recovery_action": "Original database restored to expected path.",
        "recovery_verified": "Path, schema, row count, and health verified after recovery.",
        "logging_improvement_added": "Logging requirements improved after investigation."
    }

    return mapping.get(
        event,
        "Operational event; interpretation requires surrounding evidence."
    )

required_evidence = pd.DataFrame({
    "Time": logs["timestamp"].dt.strftime("%H:%M:%S"),
    "Component": logs["component"],
    "Event": logs["event"],
    "Evidence": logs["level"].map(lambda x: f"{x} log entry"),
    "Interpretation": logs.apply(interpretation_for_event, axis=1)
})

display(required_evidence.tail(12))


# 17. Generate the Full Failure Report

The Word report uses the exact required headings from Assignment 7.2.


In [ ]:
# python-docx is usually available, but install if needed.
try:
    from docx import Document
    from docx.shared import Pt
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "python-docx", "-q"],
        check=True
    )
    from docx import Document
    from docx.shared import Pt

REPORT_FILE = BASE / "EGN321_Module7_Assignment7_2_Failure_Report_Demo.docx"

doc = Document()
doc.styles["Normal"].font.name = "Aptos"
doc.styles["Normal"].font.size = Pt(10.5)

doc.add_heading(
    "EGN 321 — Module 7 Assignment 7.2",
    0
)

doc.add_paragraph(
    "Failure Report: Diagnose, Recover, and Improve the System"
)

sections = [
    (
        "1. System Before Failure",
        "The application was operating in a HEALTHY state. "
        "The SQLite database existed, the expected readings table was available, "
        "and 20 valid readings had been stored."
    ),
    (
        "2. Failure Introduced",
        "The database file was deliberately renamed to simulate a realistic "
        "database-unavailable or incorrect-path failure."
    ),
    (
        "3. First Observable Symptom",
        f"The first observable warning/error in the available logs was "
        f"'{FIRST_OBSERVABLE}'. This is an observed symptom and is not automatically "
        f"the final root cause."
    ),
    (
        "4. Log Evidence",
        "Structured logs recorded the healthy baseline, controlled failure, "
        "database access failure, recovery action, recovery verification, and logging improvement."
    ),
    (
        "5. Timeline of Events",
        "Healthy baseline → failure injected → database access failed → "
        "recovery action → recovery verified → logging improved."
    ),
    (
        "6. Possible Cause 1",
        f"Possible cause: the expected database path became unavailable. "
        f"Deterministic path test before recovery: {cause1_before}. "
        "A missing expected path supports this hypothesis."
    ),
    (
        "7. Possible Cause 2",
        f"Possible cause: a database file exists but the required application schema "
        f"is missing or invalid. Deterministic schema test before recovery: {cause2_before}. "
        "In this controlled demo, the expected application database was unavailable."
    ),
    (
        "8. Recovery Procedure",
        "Remove any blank SQLite file created at the expected path, restore the original "
        "database file, verify the expected path, verify the readings table, and confirm preserved data."
    ),
    (
        "9. Verification of Recovery",
        f"Recovery passed: {recovery_passed}. "
        f"Rows preserved: {rows_preserved}. "
        f"Path after recovery: {cause1_after}. "
        f"Schema after recovery: {cause2_after}."
    ),
    (
        "10. What the Logs Told Me",
        "The logs established the sequence of events and confirmed that database access "
        "failed after the controlled failure. They also confirmed that recovery was attempted "
        "and that final verification passed."
    ),
    (
        "11. What the Logs Did Not Tell Me",
        "The application logs alone would not prove who or what changed the database path "
        "in a real incident. They also do not prove whether disk pressure, permissions, "
        "or another operating-system event contributed."
    ),
    (
        "12. Improvement Made",
        "Logging was improved to require component, database path, exception type, "
        "health state, attempted recovery, and verification result for future storage failures."
    ),
    (
        "13. Remaining Limitations",
        "This is a controlled classroom incident. A real deployment would also require "
        "operating-system, service-manager, storage, permission, and deployment-platform evidence."
    )
]

for heading, body in sections:
    doc.add_heading(heading, level=1)
    doc.add_paragraph(body)

doc.add_heading("Required Evidence Table", level=1)

table = doc.add_table(
    rows=1,
    cols=5
)

table.style = "Table Grid"

headers = [
    "Time",
    "Component",
    "Event",
    "Evidence",
    "Interpretation"
]

for i, h in enumerate(headers):
    table.rows[0].cells[i].text = h

for _, row in required_evidence.tail(12).iterrows():
    cells = table.add_row().cells
    for i, h in enumerate(headers):
        cells[i].text = str(row[h])

doc.add_heading("Ollama Review", level=1)
doc.add_paragraph(
    "Ollama was used after deterministic evidence had been established. "
    "Its role was to review hypotheses, identify missing evidence, propose tests, "
    "and challenge conclusions. AI output was not treated as operational evidence."
)

doc.add_paragraph("Investigator review:")
doc.add_paragraph(ollama_investigation)

doc.add_paragraph("Red-team review:")
doc.add_paragraph(ollama_red_team)

doc.save(REPORT_FILE)

print("Created:", REPORT_FILE)


# 18. Generate README.md and AI_LOG.md


In [ ]:
README_FILE = BASE / "README.md"
AI_LOG_FILE = BASE / "AI_LOG.md"

README_FILE.write_text(
f'''# EGN 321 Module 7 Assignment 7.2 Demo

## Failure Scenario
Controlled database-unavailable condition.

## Investigation Flow
Healthy baseline -> controlled failure -> structured logs -> evidence table ->
first observable symptom -> two possible causes -> deterministic tests ->
recovery -> verification -> logging improvement -> Ollama review.

## Recovery Result
Recovery passed: {recovery_passed}
Rows preserved: {rows_preserved}

## Required Evidence
- structured logs
- failure timeline
- failed-state evidence
- recovered-state evidence
- deterministic verification tests
- failure report
- updated README
- AI_LOG.md

## Important
Logs show what the application observed.
They do not automatically prove the ultimate root cause.

Ollama is a reviewer and drafting assistant.
It is not the source of operational evidence.
''',
encoding="utf-8"
)

AI_LOG_FILE.write_text(
f'''# AI_LOG.md

Tool: Ollama
Model: {OLLAMA_MODEL}

## Purpose
Review deterministic incident evidence, propose hypotheses and tests,
red-team conclusions, and assist with failure-report drafting.

## Evidence Provided
{json.dumps(evidence, indent=2)}

## Investigator Review
{ollama_investigation}

## Red-Team Review
{ollama_red_team}

## Verification Rule
AI output is not treated as evidence.
Claims must be verified using logs, SQLite, observed system behavior,
or deterministic tests.

## Instructor / Student Review
Document:
- which AI suggestions were accepted;
- which were rejected;
- which deterministic test was added;
- what changed in the final report.
''',
encoding="utf-8"
)

print("Created:", README_FILE)
print("Created:", AI_LOG_FILE)


# 19. Export All Required Assignment Deliverables

This creates a single ZIP containing:

- failure report;
- relevant log files;
- failure timeline;
- failed-condition evidence;
- recovered-condition evidence;
- updated application/evidence code;
- updated README;
- verification tests/results;
- `AI_LOG.md`;
- Ollama investigator review;
- Ollama red-team review;
- required evidence table.


In [ ]:
# Re-read final logs
records = []

with open(LOG_FILE, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

logs = pd.DataFrame(records)
logs["timestamp"] = pd.to_datetime(logs["timestamp"], utc=True)
logs = logs.sort_values("timestamp").reset_index(drop=True)

failure_timeline = logs[
    logs["event"].isin([
        "system_baseline",
        "failure_injected",
        "database_read_failed",
        "recovery_action",
        "recovery_verified",
        "logging_improvement_added"
    ])
].copy()

logs.to_csv(
    EXPORT_DIR / "relevant_log_records.csv",
    index=False
)

failure_timeline.to_csv(
    EXPORT_DIR / "failure_timeline.csv",
    index=False
)

required_evidence.to_csv(
    EXPORT_DIR / "required_evidence_table.csv",
    index=False
)

verification_results = {
    "possible_cause_1_before": cause1_before,
    "possible_cause_2_before": cause2_before,
    "possible_cause_1_after": cause1_after,
    "possible_cause_2_after": cause2_after,
    "rows_preserved": int(rows_preserved),
    "recovery_passed": bool(recovery_passed)
}

(EXPORT_DIR / "verification_results.json").write_text(
    json.dumps(verification_results, indent=2),
    encoding="utf-8"
)

(EXPORT_DIR / "failed_condition_evidence.txt").write_text(
    f'''First observable symptom: {FIRST_OBSERVABLE}

Cause 1 test before recovery:
{json.dumps(cause1_before, indent=2)}

Cause 2 test before recovery:
{json.dumps(cause2_before, indent=2)}
''',
    encoding="utf-8"
)

(EXPORT_DIR / "recovered_condition_evidence.txt").write_text(
    f'''Recovery passed: {recovery_passed}
Rows preserved: {rows_preserved}

Path test after recovery:
{json.dumps(cause1_after, indent=2)}

Schema test after recovery:
{json.dumps(cause2_after, indent=2)}
''',
    encoding="utf-8"
)

(EXPORT_DIR / "ollama_investigator_review.txt").write_text(
    ollama_investigation,
    encoding="utf-8"
)

(EXPORT_DIR / "ollama_red_team_review.txt").write_text(
    ollama_red_team,
    encoding="utf-8"
)

shutil.copy2(
    REPORT_FILE,
    EXPORT_DIR / REPORT_FILE.name
)

shutil.copy2(
    LOG_FILE,
    EXPORT_DIR / "module7_app.jsonl"
)

shutil.copy2(
    DB_FILE,
    EXPORT_DIR / "recovered_database.db"
)

shutil.copy2(
    README_FILE,
    EXPORT_DIR / "README.md"
)

shutil.copy2(
    AI_LOG_FILE,
    EXPORT_DIR / "AI_LOG.md"
)

SUBMISSION_MANIFEST = EXPORT_DIR / "SUBMISSION_MANIFEST.txt"

SUBMISSION_MANIFEST.write_text(
'''MODULE 7 ASSIGNMENT 7.2 — DEMO SUBMISSION

[1] Failure report
    EGN321_Module7_Assignment7_2_Failure_Report_Demo.docx

[2] Relevant logs
    module7_app.jsonl
    relevant_log_records.csv

[3] Failure timeline
    failure_timeline.csv

[4] Failed condition evidence
    failed_condition_evidence.txt

[5] Recovered condition evidence
    recovered_condition_evidence.txt
    recovered_database.db

[6] Verification tests/results
    verification_results.json

[7] Required evidence table
    required_evidence_table.csv

[8] Updated README
    README.md

[9] AI log
    AI_LOG.md

[10] Ollama supporting review
    ollama_investigator_review.txt
    ollama_red_team_review.txt

Student submission should also include the GitHub repository link in the LMS.
''',
encoding="utf-8"
)

ZIP_FILE = Path(
    "/content/EGN321_Module7_Assignment7_2_COMPLETE_DEMO_EXPORT.zip"
)

with zipfile.ZipFile(
    ZIP_FILE,
    "w",
    zipfile.ZIP_DEFLATED
) as z:
    for p in EXPORT_DIR.iterdir():
        if p.is_file():
            z.write(p, arcname=p.name)

print("Created:", ZIP_FILE)
print("\nFiles in submission package:")

for p in sorted(EXPORT_DIR.iterdir()):
    print(" -", p.name)

# Optional automatic download:
# from google.colab import files
# files.download(str(ZIP_FILE))


# Demo Complete

The demo now covers the full Assignment 7.2 expectation:

- exact 13 report sections;
- required evidence table;
- failure report;
- relevant logs;
- failure timeline;
- failed-state evidence;
- recovered-state evidence;
- verification tests;
- README;
- AI_LOG.md;
- Ollama evidence review;
- Ollama red-team review;
- final submission ZIP.

### Instructor closing point

> The strongest report is not the one with the most confident root-cause statement.  
> It is the one that clearly separates **evidence, hypothesis, missing evidence, verification, and remaining uncertainty**.
